In [2]:
import io
import zipfile
import httpx

import pandas as pd
from eutl_scraper.extract import (
    url_source,
    extract_accounts,
    extract_transactions,
    extract_installations,
    extract_compliance
)

## Installations

The file under *Data Download* seem to have all information we need.

In [3]:
df_inst = extract_installations()
#df_inst.info()
df_inst.query(f"installation_id == 'ES_202172'").T


,8007
registry_id,ES
registry_name,Spain
installation_name,"Aludium Transformación de Productos,S.L.U-Alic..."
account_identifier,5018274
account_registry_code,ES
account_identifier_in_reg,"Aludium Transformación de Productos, S.L.- Ali..."
eper_identification,3468
activity_type_code,27
activity_type,Production of secondary aluminium
permit_identifier,ES10030810853


## Compliance

Also the file under *Download Data* contains most information even including
details on surrendered units, allocation, and the details concerning the Swiss
registry.

In [4]:
df_comp = extract_compliance()
#df_comp.info()

## Accounts

The file from *Data Download* does not contain all data necessary. In particular, 
all firm information is missing.

In [ ]:
df_acc = extract_accounts()
df_acc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47620 entries, 0 to 47619
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   registry_id           47620 non-null  object
 1   registry_name         47620 non-null  object
 2   account_name          47620 non-null  object
 3   account_type          47620 non-null  object
 4   ets_account_type      28302 non-null  object
 5   full_type             47591 non-null  object
 6   open_date             47584 non-null  object
 7   end_of_validity_date  47620 non-null  object
 8   snapshot_date         47620 non-null  object
 9   account_id            47620 non-null  object
 10  account_type_id       47595 non-null  object
 11  closure_pending       47620 non-null  bool  
dtypes: bool(1), object(11)
memory usage: 4.0+ MB


The table that can be downloaded through the PowerBi app seems to have the missing
information on account holders. We need, however, to clarify why there are more
accounts in the "Download Data" file.

In [7]:
fn_acc = "./data/accounts_all.xlsx"
df_acc = pd.read_excel(fn_acc)
df_acc.info()

c:\GIT\eutl_scraper_v2\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47345 entries, 0 to 47344
Data columns (total 15 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Account Identifier                                   47344 non-null  object 
 1   National Administrator                               47343 non-null  object 
 2   Account Type                                         47343 non-null  object 
 3   Account Holder Name                                  47343 non-null  object 
 4   Account Name                                         47343 non-null  object 
 5   Installation/Aircraft Operator/Maritime Operator ID  22120 non-null  float64
 6   Company Registration No                              42735 non-null  object 
 7   Main Address Line                                    47334 non-null  object 
 8   City                                                 47334 non-nul

## Transactions

In [9]:
df_trans = extract_transactions()
df_trans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1875818 entries, 0 to 1875817
Data columns (total 65 columns):
 #   Column                                                   Dtype  
---  ------                                                   -----  
 0   TRANSACTION_ID                                           object 
 1   TRANSACTION_TYPE                                         object 
 2   TRANSACTION_DATE                                         object 
 3   TRANSACTION_STATUS                                       object 
 4   TRANSFERRING_REGISTRY_NAME                               object 
 5   TRANSFERRING_ACCOUNT_TYPE                                float64
 6   TRANSFERRING_ACCOUNT_TYPE2                               object 
 7   TRANSFERRING_ACCOUNT_OPEN_DT                             object 
 8   TRANSFERRING_ACCOUNT_END_OF_VALIDITY                     object 
 9   TRANSFERRING_ACCOUNT_NAME                                object 
 10  TRANSFERRING_ACCOUNT_IDENTIFIER           

In [3]:
df.columns

Index(['TRANSACTION_ID', 'TRANSACTION_TYPE', 'TRANSACTION_DATE',
       'TRANSACTION_STATUS', 'TRANSFERRING_REGISTRY_NAME',
       'TRANSFERRING_ACCOUNT_TYPE', 'TRANSFERRING_ACCOUNT_TYPE2',
       'TRANSFERRING_ACCOUNT_OPEN_DT', 'TRANSFERRING_ACCOUNT_END_OF_VALIDITY',
       'TRANSFERRING_ACCOUNT_NAME', 'TRANSFERRING_ACCOUNT_IDENTIFIER',
       'TRANSFERRING_ACCOUNT_HOLDER', 'TRANSFERRING_ACCOUNT_HOLDER_ADDRESS1',
       'TRANSFERRING_ACCOUNT_HOLDER_ADDRESS2',
       'TRANSFERRING_ACCOUNT_HOLDER_CITY',
       'TRANSFERRING_ACCOUNT_HOLDER_POSTAL_CODE',
       'TRANSFERRING_ACCOUNT_HOLDER_COUNTRY_CODE',
       'TRANSFERRING_ACCOUNT_HOLDER_COMPANY_REGISTRATION_NUMBER',
       'TRANSFERRING_ACCOUNT_HOLDER_LEI', 'TRANSFERRING_INSTALLATION_NAME',
       'TRANSFERRING_INSTALLATION_INSTALLATION_IDENTIFIER',
       'TRANSFERRING_INSTALLATION_PERMIT_IDENTIFIER',
       'TRANSFERRING_INSTALLATION_PARENT_COMPANY',
       'TRANSFERRING_INSTALLATION_SUBSIDIARY_COMPANY',
       'TRANSFERRING_INSTALLA

In [10]:
map_col = {
    "REGISTRY_CODE": "registry_id",
}
df = (
        df_comp
        .assign(
            installation_id=lambda df: df.REGISTRY_CODE
            + "_"
            + df.INSTALLATION_IDENTIFIER.astype(str)
        )
        .drop(columns=["INSTALLATION_IDENTIFIER"])
        .rename(columns=map_col)
        .rename(columns=lambda x: x.lower())
    )

In [8]:
df_comp.head()

,INSTALLATION_IDENTIFIER,REGISTRY_CODE,REGISTRY_NAME,INSTALLATION_NAME,PERIOD_YEAR,VERIFIED_EMISSIONS,CH_VERIFIED_EMISSIONS,ALLOCATION,ALLOCATION_RES,ALLOCATION_TRA,CH_ALLOCATION,EXCLUDED,CH_EXCLUDED,SURR_EUA,SURR_EUAA,SURR_CHU,SURR_CHUA,SURR_ERU_FROM_AAU,SURR_FORMER_EUA,SURR_CER,SURR_ALLOWANCE_CP0,SURR_ALL,SNAPSHOT_DATE
0,1,AT,Austria,Calmit Bad Ischl,2005.0,41399.0,-1.0,44894.0,0.0,0.0,-1.0,NO,NO,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,41399.0,41399.0,2025-10-13
1,1,AT,Austria,Calmit Bad Ischl,2025.0,-1.0,-1.0,33216.0,0.0,0.0,-1.0,NO,NO,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,2025-10-13
2,1,AT,Austria,Calmit Bad Ischl,2007.0,51041.0,-1.0,44894.0,0.0,0.0,-1.0,NO,NO,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,51041.0,51041.0,2025-10-13
3,1,AT,Austria,Calmit Bad Ischl,2008.0,57009.0,-1.0,43171.0,0.0,0.0,-1.0,NO,NO,-1.0,-1.0,-1.0,-1.0,-1.0,57009.0,-1.0,-1.0,57009.0,2025-10-13
4,1,AT,Austria,Calmit Bad Ischl,2009.0,44299.0,-1.0,43171.0,0.0,0.0,-1.0,NO,NO,-1.0,-1.0,-1.0,-1.0,-1.0,44299.0,-1.0,-1.0,44299.0,2025-10-13
